In [1]:
%cd lib

/home/grader-cse255-01/public/notebooks/Section2-PCA/PCA/lib


In [2]:
!ls

MultiPlot.py		 decomposer.py	    row_parser.py
Reconstruction_plots.py  getFiles.py	    sparkConfig.py
YearPlotter.py		 import_modules.py  spark_PCA.py
__init__.py		 leaflet.py	    spark_PCA_HW.py
__pycache__		 lib.tgz	    start_spark_context.py
binary_search.py	 numpy_pack.py	    startup.py
computeStatistics.py	 old


In [11]:
!ls -lrt ../*.py

-rw-r--r-- 1 grader-cse255-01 root 0 Apr 13  2023 ../__init__.py


In [ ]:
%%writefile YearPlotter.py
from datetime import date
from numpy import shape
from matplotlib.dates import MonthLocator, DateFormatter
class YearPlotter:
    def __init__(self):
        start=365*1+1
        self.dates=[date.fromordinal(i) for i in range(start,start+365)]
        self.monthsFmt = DateFormatter("%b")
        self.months = MonthLocator(range(1, 13), bymonthday=1, interval=3)
        #self.i=0

    def plot(self,T,fig,ax,label='',labels=None,title=None,last=365):
        
        shp=shape(T)
        if shp[0] != 366:
            raise ValueError("First dimension of T should be 366. Shape(T)="+str(shape(T)))
        if len(shp)==1:
            ax.plot(self.dates,T[:last],label=label);
        else:
            if labels is None:
                labels=[str(i) for i in range(shp[1])]
            for i in range(shp[1]):
                ax.plot(self.dates,T[:last,i],label=labels[i])
        ax.xaxis.set_major_locator(self.months)
        ax.xaxis.set_major_formatter(self.monthsFmt)
        if not title is None:
            ax.set_title(title)
        #rotate and align the tick labels so they look better
        fig.autofmt_xdate()
        ax.grid()
        ax.legend()
        #ax.set_xlim([dates_to_plot[0], dates_to_plot[-1]])  # Always show full year
        ax.xaxis.set_major_locator(self.months)
        ax.xaxis.set_major_formatter(self.monthsFmt)
       


In [14]:
%%writefile startup.py
# Prepare python libraries for distribution to executors
# !tar -czvf lib.tgz lib/*.py

import sys

if len(sys.argv)<=1:
    sc_type='S'
else:
    sc_type=sys.argv[1]
print('sc_type=',sc_type)

# start sparkContext
import pandas as pd
import numpy as np
import sklearn as sk
import urllib
import math

import pyspark
from pyspark import SparkContext
from lib import sparkConfig

from start_spark_context import start_spark_context, get_current_namespace

if sc_type!='S':
    sc = SparkContext('local[10]')
else:
    sc=start_spark_context()

print('sparkContext=',sc)
print()

# start sqlContext
from pyspark.sql import *
import pyspark.sql
sqlContext = SQLContext(sc)
import numpy as np

#load libraries to workers
sc.addPyFile("lib/numpy_pack.py")
sc.addPyFile("lib/spark_PCA.py")
sc.addPyFile("lib/computeStatistics.py")
sc.addPyFile("lib/decomposer.py")

import warnings  # Suppress Warnings
warnings.filterwarnings('ignore')
sc.setLogLevel("ERROR")

_figsize=(10,7)

### Load lib archive
#sc.addArchive("lib.tgz")  # extract directory on all workers

### Load the required libraries

from lib.YearPlotter import YearPlotter
#from lib.decomposer import *
#from lib.Reconstruction_plots import *

#from lib.import_modules import import_modules,modules
#import_modules(modules)

# import widgets library
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, fixed, interact_manual,widgets
import ipywidgets as widgets
print('version of ipwidgets=',widgets.__version__)

import warnings  # Suppress Warnings
warnings.filterwarnings('ignore')

## Change the paths here to account for current location of parquest files
## load measurement and stations dataframe
ns=get_current_namespace()
parquet_root=f'/home/{ns}/public/Data/weather'
print('parquet_root=',parquet_root)

measurements_path=parquet_root+'/weather-parquet'
measurements=sqlContext.read.parquet(measurements_path)
sqlContext.registerDataFrameAsTable(measurements,'measurements')

print('measurements is a Dataframe (and table) with %d records'%(measurements.count()))

stations_path=parquet_root+'/stations-parquet'
stations=sqlContext.read.parquet(stations_path)
sqlContext.registerDataFrameAsTable(stations,'stations')
print('stations is a Dataframe (and table) with %d records'%(stations.count()))

weather=measurements.join(stations,on='station')
print('weather is a Dataframe (and table) which is a join of measurements and stations with %d records'%(weather.count()))
sqlContext.registerDataFrameAsTable(weather,'weather')


Overwriting startup.py


In [15]:
%%writefile numpy_pack.py
import numpy as np
"""Code for packing and unpacking a numpy array into a byte array.
   the array is flattened if it is not 1D.
   This is intended to be used as the interface for storing 
   
   This code is intended to be used to store numpy array as fields in a dataframe and then store the 
   dataframes in a parquet file.
"""

def packArray(a):
    """
    pack a numpy array into a bytearray that can be stored as a single 
    field in a spark DataFrame

    :param a: a numpy ndarray 
    :returns: a bytearray
    :rtype:

    """
    if type(a)!=np.ndarray:
        raise Exception("input to packArray should be numpy.ndarray. It is instead "+str(type(a)))
    return bytearray(a.tobytes())

def unpackAndScale(row):
    """ Unpack bytearray, then if measurement is in ['TMIN','TMAX','TOBS']
    then divide by 10 to get celsius
    """
    #from numpy_pack import unpackArray

    if '_S' in row.Measurement:
        v=unpackArray(row.Values,data_type=np.float16)
    else:
        v=unpackArray(row.Values)
    if(row.Measurement in ['TMIN','TMAX','TOBS']):
        v=v/10
    return v

def unpackArray(x,data_type=np.int16):
    """
    unpack a bytearray into a numpy.ndarray, values Smaller than -990 (nominally -999) are mapped to np.nan

    :param x: a bytearray
    :param data_type: The dtype of the array. This is important because if determines how many bytes go into each entry in the array.
    :returns: a numpy array of float16
    :rtype: a numpy ndarray of dtype data_type.

    """
    V=np.frombuffer(x,dtype=data_type)
    V=np.array(V,dtype=np.float16)
    V[V<-990]=np.nan
    return V


Overwriting numpy_pack.py


In [16]:
!pwd

/home/grader-cse255-01/public/notebooks/Section2-PCA/PCA/lib


In [18]:
%%writefile decomposer.py
import numpy as np
import os, sys

from pyspark.sql import Row

class Eigen_decomp:
    """A class for approximating a function with an orthonormal set of
    functions    """
    
    def __init__(self,x,f,mean,U):
        """ Initialize the widget

        :param x: defines the x locations
        :param f: the function to be approximated
        :param mean: The initial approximation (the mean)
        :param U: an orthonormal matrix with m columns (number of vectors to use in decomposition)
        :returns: None
        """

        self.U=U
        
        self.x=x
        self.mean=mean

        self.f=f
        self.startup_flag=True
        self.C=np.dot((np.nan_to_num(f-mean)),self.U)  # computer the eigen-vectors coefficients.
        self.C=np.array(self.C).flatten()
        self.m,self.n=np.shape(self.U)
        self.coeff={'c'+str(i):self.C[i] for i in range(self.C.shape[0])} # Put the coefficient in the dictionary format that is used by "interactive".
        return None

    def compute_var_explained(self):
        """Compute a summary of the decomposition

        :returns: ('total_energy',total_energy),
                ('residual var after mean, eig1,eig2,...',residual_var[0]/total_energy,residual_var[1:]/residual_var[0]),
                ('reduction in var for mean,eig1,eig2,...',percent_explained[0]/total_energy,percent_explained[1:]/residual[0]),
                ('eigen-vector coefficients',self.C)

        :rtype: tuple of pairs. The first element in each pair is a
        description, the second is a number or a list of numbers or an
        array.

        """
        def compute_var(vector):
            v=np.array(np.nan_to_num(vector),dtype=np.float64)
            return np.dot(v,v) # /float(total_variance)
        
        k=self.U.shape[1]
        residual_var=np.zeros(k+1)
        
        residual=self.f   # init residual to function 
        total_energy=compute_var(residual)
        residual=residual-self.mean # set residual to function - mean 
        residual_var[0]=compute_var(residual)
        # compute residuals after each 
        for i in range(k):
            g=self.U[:,i]*self.coeff['c'+str(i)]
            g=np.array(g).flatten()
            residual=residual-g # subtract projection on i'th coefficient from residual
            residual_var[i+1]=compute_var(residual)

        # normalize residuals
        _residuals=residual_var/(residual_var[0]+1e-10)   # Divide ressidulas by residuals after subtracting mean
        _residuals[0] = residual_var[0]/(total_energy+1e-10)

        return (('total_energy',total_energy),
                ('fraction residual var after mean, eig1,eig2,...',_residuals),
                ('eigen-vector coefficients',self.coeff))

    # total_var,residuals,reductions,coeff=recon.compute_var_explained()


def decompose_dataframe(sc,sqlContext,df,EigVec,Mean):
    """ run decompose(row) on all rows of a dataframe, return an augmented dataframe with columns
    corresponding to residuals and coefficients.
    """
    def decompose(row):
        """compute residual and coefficients for a single row      

        :param row: SparkSQL Row that contains the measurements for a particular station, year and measurement. 
        :returns: the input row with additional information from the eigen-decomposition.
        :rtype: SparkSQL Row 

        Note that Decompose is designed to run inside a spark "map()" command inside decompose_dataframe.
        Mean and EigVec are sent to the workers as global variables of "decompose"

        """
        #from numpy_pack import unpackAndScale
        import numpy_pack
        Series=np.array(numpy_pack.unpackAndScale(row),dtype=np.float64)
        #Mean=Mean_BC.value
        #EigVec=EigVec_BC.value
        recon=Eigen_decomp(None,Series,Mean,EigVec);
        total_var,residuals,coeff=recon.compute_var_explained()

        D=row.asDict()
        D['total_var']=float(total_var[1])
        D['res_mean']=float(residuals[1][0])
        for i in range(1,residuals[1].shape[0]):
            D['res_'+str(i)]=float(residuals[1][i])
            D['coeff_'+str(i)]=float(coeff[1]['c'+str(i-1)])
        return Row(**D)

    # main of decompose_dataframe
    
    #EigVec_BC=sc.broadcast(EigVec)
    #Mean_BC=sc.broadcast(Mean)
    import decomposer
    rdd2=df.rdd.map(decompose).cache()
    rdd2.count()
    return sqlContext.createDataFrame(rdd2)


Overwriting decomposer.py


In [19]:
!pwd

/home/grader-cse255-01/public/notebooks/Section2-PCA/PCA/lib


In [ ]:
!ls ../*.py

In [ ]:
%%writefile computeStatistics.py

from numpy import linalg as LA
import numpy as np

from time import time
from pickle import load,dump

from numpy_pack import packArray,unpackArray,unpackAndScale
from spark_PCA import computeCov


_measurements=['TMAX', 'SNOW', 'SNWD', 'TMIN', 'PRCP', 'TOBS']
_measurements=_measurements+[x+'_S10' for x in _measurements] + [x+'_S20' for x in _measurements]
_measurements

def load_or_compute_statistics(sc,sqlContext,pkl_filename,weather_df,ms):
    import os
    if os.path.isfile(pkl_filename):   
        print('precomputed statistics file exists')
        with open(pkl_filename,'br') as pkl_file:
            stat=load(pkl_file)
    else:
        print('computing statistics')
        stat=computeStatistics(sc,sqlContext,weather_df,measurements=ms)
        with open(pkl_filename,'bw') as pkl_file:
            dump(stat,pkl_file)
    return stat


def computeStatistics(sqlContext,df,measurements=_measurements):
    """Compute all of the statistics for a given dataframe
    Input: sqlContext: to perform SQL queries
            df: dataframe with the fields 
            Station(string), Measurement(string), Year(integer), Values (byteArray with 366 float16 numbers)
    returns: STAT, a dictionary of dictionaries. First key is measurement, 
             second keys described in computeStats.STAT_Descriptions
    """
    print('computestatistics loading')
 
    sqlContext.registerDataFrameAsTable(df,'local_weather')
    STAT={}  # dictionary storing the statistics for each measurement
    
    for meas in measurements:
        t=time()
        Query="SELECT * FROM local_weather\n\tWHERE measurement = '%s'"%(meas)
        mdf = sqlContext.sql(Query)
        mdf_count=mdf.count()
        print(meas,': shape of mdf is ',mdf_count)
        if mdf_count==0:
              continue

        data=mdf.rdd.map(lambda row: unpackAndScale(row)).cache()
        data.count()

        #Compute basic statistics
        STAT[meas]=computeOverAllDist(data)   # Compute the statistics 

        # compute covariance matrix
        OUT=computeCov(data)

        #find PCA decomposition
        eigval,eigvec=LA.eig(OUT['Cov'])

        # collect all of the statistics in STAT[meas]
        STAT[meas]['eigval']=eigval
        STAT[meas]['eigvec']=eigvec
        STAT[meas].update(OUT)

        print('time for',meas,'is',time()-t)
    
    return STAT


def find_percentiles(SortedVals,percentile):
    L=int(len(SortedVals)/percentile)
    return SortedVals[L],SortedVals[-L]
  
def computeOverAllDist(rdd0):
    # Compute a sample of the number of nan per year
    UnDef=np.array(rdd0.map(lambda row:sum(np.isnan(row))).sample(False,0.1).collect())
    flat=rdd0.flatMap(lambda v:list(v)).filter(lambda x: not np.isnan(x)).cache()
    # compute first and second order statistics
    try:
        count,S1,S2=flat.map(lambda x: np.float64([1,x,x**2]))\
                      .reduce(lambda x,y: x+y)
    except:
        print('error in computeOverallDist, len rdd0=',rdd.count())
        print('undefs=',UnDef.take(10))
        print('flat=',flat.take(10))
    mean=S1/count
    std=np.sqrt(S2/count-mean**2)
    
    #Sample 0.01 percent of the values and generate a sorted array that can be used to plot an approximate CDF
    Vals=flat.sample(False,0.0001).collect()
    SortedVals=np.array(sorted(Vals))
    low100,high100=find_percentiles(SortedVals,100)
    low1000,high1000=find_percentiles(SortedVals,1000)
    return {'UnDef':UnDef,\
          'mean':mean,\
          'std':std,\
          'SortedVals':SortedVals,\
          'low100':low100,\
          'high100':high100,\
          'low1000':low1000,\
          'high1000':high1000
          }

# description of data returned by computeOverAllDist
STAT_Descriptions=[
 # distribution of undefined/defined entries
 ('UnDef', 'sample of number of undefs per row', 'vector whose length varies between measurements'),
 ('NE', 'count of defined values per day', (366,)),

 # statistics of scalars, taken over all entries in all vectors
 ('SortedVals', 'Sample of values', 'vector whose length varies between measurements'),
 ('mean', 'mean value', ()),
 ('std', 'std', ()),
 ('low100', 'bottom 1%', ()),
 ('high100', 'top 1%', ()),
 ('low1000', 'bottom 0.1%', ()),
 ('high1000', 'top 0.1%', ()),
    
 # statistics of yearly measurement vectors   
 ('E', 'Sum of values per day', (366,)),
 ('Mean', 'E/NE', (366,)),
 ('O', 'Sum of outer products', (366, 366)),
 ('NO', 'counts for outer products', (366, 366)),
 ('Cov', 'O/NO', (366, 366)),
 ('Var', 'The variance per day = diagonal of Cov', (366,)),
 ('eigval', 'PCA eigen-values', (366,)),
 ('eigvec', 'PCA eigen-vectors', (366, 366))
]


In [ ]:
!pwd

In [6]:
%%writefile spark_PCA.py
import numpy as np
from numpy import linalg as LA

def outerProduct(X):
    """Computer outer product and indicate which locations in matrix are undefined"""
    O=np.outer(X,X)
    N=1-np.isnan(O)
    return (O,N)

def sumWithNan(M1,M2):
    """Add two pairs of (matrix,count)"""
    (X1,N1)=M1
    (X2,N2)=M2
    N=N1+N2
    X=np.nansum(np.dstack((X1,X2)),axis=2)
    return (X,N)

def computeCov(RDDin):
    """computeCov recieves as input an RDD of np arrays, all of the same length, 
    and computes the covariance matrix for that set of vectors"""
    RDD=RDDin.map(lambda v:np.array(np.insert(v,0,1),dtype=np.float64)) # insert a 1 at the beginning of each vector so that the same 

    
    from spark_PCA import outerProduct, sumWithNan
    
    #calculation also yields the mean vector
    OuterRDD=RDD.map(outerProduct)   # separating the map and the reduce does not matter because of Spark uses lazy execution.
    (S,N)=OuterRDD.reduce(sumWithNan)

    E=S[0,1:]
    NE=np.float64(N[0,1:])
    #print('shape of E=',E.shape,'shape of NE=',NE.shape)
    Mean=E/NE
    O=S[1:,1:]
    NO=np.float64(N[1:,1:])
    Cov=O/NO - np.outer(Mean,Mean)
    # Output also the diagnal which is the variance for each day
    Var=np.array([Cov[i,i] for i in range(Cov.shape[0])])
    return {'E':E,'NE':NE,'O':O,'NO':NO,'Cov':Cov,'Mean':Mean,'Var':Var}

if __name__=="__main__":
    # create synthetic data matrix with j rows and rank k
    
    from pyspark import SparkContext
    sc=SparkContext()
    
    V=2*(np.random.random([2,10])-0.5)
    data_list=[]
    for i in range(1000):
        f=2*(np.random.random(2)-0.5)
        data_list.append(np.dot(f,V))
    # compute covariance matrix
    RDD=sc.parallelize(data_list)
    OUT=computeCov(RDD)

    #find PCA decomposition
    eigval,eigvec=LA.eig(OUT['Cov'])
    print('eigval=',eigval)
    print('eigvec=',eigvec)

Overwriting spark_PCA.py


In [6]:
!pwd

/home/grader-cse255-01/public/notebooks/Section2-PCA/PCA/lib


In [7]:
%cd ..

/home/grader-cse255-01/public/notebooks/Section2-PCA/PCA


In [8]:
!ls -lrt *.py

-rw-r--r-- 1 grader-cse255-01 root 0 Apr 13  2023 __init__.py


In [53]:
!rm numpy_pack.py

In [60]:
!ls -lrt lib

total 220
-rw-r--r-- 1 grader-cse255-01 root  1989 Apr 13  2023 spark_PCA_HW.py
-rw-r--r-- 1 grader-cse255-01 root   749 Apr 13  2023 sparkConfig.py
-rw-r--r-- 1 grader-cse255-01 root  2567 Apr 13  2023 row_parser.py
-rw-r--r-- 1 grader-cse255-01 root 19168 Apr 13  2023 leaflet.py
-rw-r--r-- 1 grader-cse255-01 root  1121 Apr 13  2023 import_modules.py
-rw-r--r-- 1 grader-cse255-01 root  1397 Apr 13  2023 getFiles.py
-rw-r--r-- 1 grader-cse255-01 root   608 Apr 13  2023 binary_search.py
-rw-r--r-- 1 grader-cse255-01 root     0 Apr 13  2023 __init__.py
-rw-r--r-- 1 grader-cse255-01 root   413 Apr 13  2023 MultiPlot.py
drwxr-xr-x 2 grader-cse255-01 root     6 Apr  6 21:40 old
lrwxrwxrwx 1 grader-cse255-01 root    10 Apr 16 00:42 lib.tgz -> ../lib.tgz
-rw-r--r-- 1 grader-cse255-01 root   789 Apr 16 00:54 start_spark_context.py
-rw-r--r-- 1 grader-cse255-01 root  1862 Apr 22 18:25 spark_PCA.py
-rw-r--r-- 1 grader-cse255-01 root  1364 Apr 23 22:52 YearPlotter.py
-rw-r--r-- 1 grader-cse255-01

In [61]:
!ls -l *.py

-rw-r--r-- 1 grader-cse255-01 root 0 Apr 13  2023 __init__.py


In [62]:
%cd lib

/home/grader-cse255-01/public/notebooks/Section2-PCA/PCA/lib
